In [11]:

#################################################################################################################################################################################################################################################################################
#CONEXION AL ORACLE

import oracledb
import pandas as pd
from sqlalchemy import create_engine
import sys
from sqlalchemy import create_engine
from sqlalchemy import text
import sys
from decouple import config

oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine2 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection2 = engine2.connect()

base2 = pd.read_sql_query(f"SELECT * FROM cmcsp10", con=connection2)

connection2.close()




In [12]:
tabla='cmcsp10'
dim='dim_csep'

In [13]:
base2.columns

Index(['csecod', 'csedes', 'cseedades', 'cseedahas', 'indsexadmcod',
       'csetipcod', 'cseestregcod', 'cseanio', 'csectlapeflg', 'csectlgesflg'],
      dtype='object')

In [14]:
#################################################################################################################################################################################################################################################################################
#CREAMOS LA TABLA TEMPORAL Y LA SUBIMOS AL POSTGRES SQL

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()


query_count_before = "SELECT COUNT(*) FROM cmcsp10"
cant_antes = connection3.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla cmcsp10 antes de la inserción: {cant_antes}")


#connection3.execute('CREATE TEMPORARY TABLE tmp_cmcsp10 ()')
base2.to_sql(name='tmp_cmcsp10', con=engine3, if_exists='replace', index=False)

#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL cmcsp10 FALSO CON LO OBTENIDO DEL ORACLE

query="""
ALTER TABLE tmp_cmcsp10
ALTER COLUMN csecod  TYPE character(2),
ALTER COLUMN csedes  TYPE character(120),
ALTER COLUMN cseedades  TYPE numeric(3,0),
ALTER COLUMN cseedahas  TYPE numeric(3,0);

UPDATE cmcsp10
SET csecod = t.csecod,
csedes = t.csedes,
cseedades =t.cseedades,
cseedahas =t.cseedahas

FROM tmp_cmcsp10 t 
WHERE cmcsp10.csecod = t.csecod AND TRIM(cmcsp10.csecod) <> '' AND cmcsp10.csecod IS NOT NULL;

INSERT INTO cmcsp10 
(csecod,csedes,cseedades,cseedahas) 
SELECT 
csecod,csedes,cseedades,cseedahas

FROM tmp_cmcsp10  t
WHERE NOT EXISTS (
    SELECT 1 
    FROM cmcsp10 
    WHERE cmcsp10.csecod = t.csecod and cmcsp10.csecod IS NOT NULL
) ;


"""

c1= text(query)
connection3.execute(c1)

#BORRAMOS LAS TABLAS
query2="""
DROP TABLE tmp_cmcsp10;
SELECT COUNT(*) FROM cmcsp10;
"""


c2= text(query2)
cursor=connection3.execute(c2)
cant_despues = cursor.fetchone()[0]
print(f"Cantidad de filas en la tabla cmcsp10 después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")
base2 = pd.read_sql_query(f"SELECT * FROM cmcsp10", con=connection3)


Cantidad de filas en la tabla cmcsp10 antes de la inserción: 27
Cantidad de filas en la tabla cmcsp10 después de la inserción: 27
La cantidad de filas insertadas fue de: 0


In [15]:

#################################################################################################################################################################################################################################################################################

#SUBIMOS ESA TABLA ACTUALIZADA COMO TEMPORAL A LA BASE DEL DIM ACTIVIT

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="INDICADORES_ESSALUD"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine4 = create_engine(cadena4)
connection4 = engine4.connect()


base2.to_sql(name='tmp_cmcsp10', con=engine4, if_exists='replace', index=False)

#SUBIMOS ESA TABLA ACTUALIZADA COMO TEMPORAL A LA BASE DEL DIM ACTIVIT
base2.to_sql(name=f'tmp_{tabla}', con=engine4, if_exists='replace', index=False)

#ACTUALIZAMOS EL DIM ACTIVITI FALSO CON LA BASE DE DATOS QUE SUBIMOS
query_count_before = f"SELECT COUNT(*) FROM {dim}"
cant_antes = connection4.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla {dim} antes de la inserción: {cant_antes}")

query=f"""

ALTER TABLE tmp_{tabla} 
ALTER COLUMN csecod  TYPE character(2),
ALTER COLUMN csedes  TYPE character(120),
ALTER COLUMN cseedades  TYPE numeric(3,0),
ALTER COLUMN cseedahas  TYPE numeric(3,0);

INSERT INTO {dim} 

(cod_cse,des_cse,edad_des,edad_has) 

SELECT 

csecod,csedes,cseedades,cseedahas

FROM tmp_{tabla} t
WHERE NOT EXISTS (
    SELECT 1 
    FROM {dim} d 
    WHERE 
    
    d.cod_cse = t.csecod AND
    d.des_cse = t.csedes AND
    d.edad_des = t.cseedades AND
    d.edad_has = t.cseedahas
);
"""



c1= text(query)
cursor=connection4.execute(c1)

#BORRAMOS LAS TABLAS
query2=f"""
DROP TABLE tmp_{tabla};
SELECT COUNT(*) FROM {dim};
"""

c2= text(query2)
cursor=connection4.execute(c2)
cant_despues = cursor.fetchone()[0]
print(f"Cantidad de filas en la tabla {dim} después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

Cantidad de filas en la tabla dim_csep antes de la inserción: 27
Cantidad de filas en la tabla dim_csep después de la inserción: 27
La cantidad de filas insertadas fue de: 0
